# Source Notebook Template
### raw → stg | VMI Analytics Platform

---

## Overview

This notebook ingests a single source file, validates it, and produces two governed 
Delta tables — a raw snapshot and a cleaned staged version — along with a pipeline 
run log entry.

---

## Pseudocode

    - Load libraries and configuration
    - Check source file exists and is fresh
    - Read source file into memory as-is
    - Validate raw structure — columns, row count, nulls
    - Stamp audit columns and write raw Delta table (append)
    - Apply standardization — renames, type casting, null handling, deduplication
    - Validate staged output — business key, required fields, row count
    - Write staged Delta table (append)
    - Log run result to pipeline_runs table
    - Raise any failures visibly — never suppress errors silently

## Outputs

Table names are auto-generated from the `DOMAIN` and `SOURCE_NAME` values set in 
Section 2 (Configuration). Update those two values and the table names below resolve 
automatically — do not rename tables manually.

| Table | Layer | Description |
| --- | --- | --- |
| `raw.{DOMAIN}_{SOURCE_NAME}` | raw | Unmodified source snapshot with audit columns |
| `stg.{DOMAIN}_{SOURCE_NAME}` | stg | Cleaned, standardized and deduplicated source data |
| `log.pipeline_runs` | log | One run record per execution — success or failure |

**Example — Johnson Brothers distributor inventory:**

| Table | Layer | Description |
| --- | --- | --- |
| `raw.cust_inventory_johnson_brothers` | raw | Unmodified source snapshot with audit columns |
| `stg.cust_inventory_johnson_brothers` | stg | Cleaned, standardized and deduplicated source data |
| `log.pipeline_runs` | log | One run record per execution — success or failure |

**Example — Oracle on-hand inventory:**

| Table | Layer | Description |
| --- | --- | --- |
| `raw.oracle_inventory_on_hand` | raw | Unmodified source snapshot with audit columns |
| `stg.oracle_inventory_on_hand` | stg | Cleaned, standardized and deduplicated source data |
| `log.pipeline_runs` | log | One run record per execution — success or failure |

---

> **To repurpose this template:** update the **Configuration** and **Transform & Standardize** 
> sections only. All other sections should require no changes for a standard CSV or Excel source.


## 1 · Imports

Standard libraries required across all source notebooks. Do not add source-specific 
imports here — if a one-off library is needed for a specific source, note it clearly 
at the top of the section where it is used.

**Library Guide:**

- **`pandas`** — Primary data manipulation library. Used for reading source files, 
applying transformations and constructing log entries.

- **`numpy`** — Numerical operations and null handling support. Used alongside pandas 
for type casting and data quality checks.

- **`math`** — Standard Python math library. Used specifically for `math.ceil()` to 
round quantity fields up to the nearest whole integer per governed rounding standards.

- **`os`** — Operating system interface. Used to check file existence and retrieve 
file modification timestamps for freshness validation.

- **`datetime`, `date`** — Date and timestamp handling for audit columns and run 
metadata.

- **`uuid`** — Generates a unique `pipeline_run_id` for each execution, tying all 
records and log entries back to a specific run.

- **`traceback`** — Captures full error stack traces in exception handling so failures 
are logged with enough detail to diagnose without re-running.

- **`pyspark`** — Fabric's distributed compute engine. Used specifically for writing 
Delta tables — pandas DataFrames are converted to Spark DataFrames at write time.

- **`getpass`** — Fallback user identity retrieval for local or non-Spark execution 
contexts. Used in combination with a Spark SQL call to resolve the current executing 
user for pipeline run logging. In Fabric production, the Spark method takes precedence.

- **`pytz`** — Timezone handling library. Used to convert pipeline run timestamps 
from UTC (Fabric compute default) to the team's local timezone (US/Eastern) so 
log entries reflect accurate local execution time.


In [ ]:
# =============================================================================
# NOTEBOOK:  nbook_template_source
# LAYER:     raw → stg
# DOMAIN:    [domain]
# SOURCE:    [source_name]
# AUTHOR:    [your name]
# CREATED:   [date]
# MODIFIED:  [date]
# DESC:      Ingests [source] data, writes raw_ and stg_ Delta tables.
# =============================================================================

import pandas as pd
import numpy as np
import math
import os
import traceback
import uuid
import getpass
from datetime import datetime, date
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import pyspark.sql.functions as F
from datetime import datetime, date, timezone
import pytz


StatementMeta(, , -1, SessionStarting, , SessionStarting, True)

## 2 · Configuration

> Update all values in this block before running. No hardcoded values should appear outside this section.

This block is the single source of configuration for the entire notebook. All source-specific 
settings, table names, schema expectations and business rules are defined here and referenced 
throughout. When repurposing this template for a new source, this cell and the transformation 
logic in Section 7 are the only sections that should require meaningful changes.

**Section Guide:**

- **Run Metadata** — Auto-generated values that tag every record and log entry to a specific 
pipeline execution. Do not modify these manually.

- **Source Identity** — Defines the source name, domain and notebook type. 
`SOURCE_NAME` must be lowercase snake_case and match the naming convention 
(e.g. `johnson_brothers`). `NOTEBOOK_TYPE` must be set to `source`, `core` 
or `output` — this determines how the pipeline run log entry is structured. 
This value is used to auto-generate table names.

---

> ⚠️ **Important — Always Start in Dev**
>
> The `ENV` variable controls whether this notebook writes to sandbox tables or 
> governed production tables. It must be set to `"dev"` for all development, 
> testing and validation work. Do not change it to `"prod"` until the notebook 
> has been fully tested, validated against expected output and reviewed by the 
> platform owner. Writing bad data to a governed production table requires a 
> deliberate Delta maintenance operation to correct — it is not a simple undo. 
> When in doubt, leave it as `"dev"`.

---

- **Source File** — The full path to the source file and its format. For L drive files, use 
the full UNC path. `SRC_SHEET` is only required for Excel files — leave as `None` for CSVs.

- **Target Tables** — Auto-generated from `SOURCE_NAME` and layer prefixes. Do not rename 
manually — update `SOURCE_NAME` and these will resolve correctly.

- **Schema Expectations** — `EXPECTED_COLUMNS` defines all columns the source file should 
contain. `REQUIRED_COLUMNS` is the subset that must never be null. Both are used in raw 
validation (Section 5) to catch schema drift before it causes silent failures downstream.

- **Business Key** — The combination of columns that uniquely identifies one row in this 
source. Used for deduplication in the staged layer. Must be explicitly defined — 
`drop_duplicates()` with no key is not acceptable per platform standards.

- **Column Renames** — Maps raw source column names to platform snake_case standards. 
All renames are applied in Section 7 during transformation. Adding a rename here is the 
correct way to handle source column name changes — do not patch column names inline.

#### `cust_inventory` Domain — Standard Column Rename Targets
Right-side values in `COLUMN_RENAMES` must match these names so all sources union cleanly in the core layer.

| Renamed Column | Represents | Required? |
|---|---|---|
| `location` | Distributor location or warehouse code | Yes |
| `item` | Distributor-facing item / SKU code | Yes |
| `on_hand_qty` | On-hand inventory quantity | Yes |
| `in_transit_qty` | In-transit quantity | If available in source |
| `max_order_qty` | Maximum order quantity | If available in source |

- **Freshness** — Defines acceptable staleness thresholds for this source. A warning is 
logged if the source file is older than `FRESHNESS_WARN_HOURS`. The pipeline fails if older 
than `FRESHNESS_FAIL_HOURS`. `SOURCE_OWNER` is the contact if freshness validation fails.

**Freshness Threshold Guide:**

Thresholds should reflect the expected delivery cadence plus a reasonable buffer. 
The warn threshold should fire before a late file becomes a pipeline-breaking problem. 
The fail threshold should stop the pipeline before stale data reaches a governed table.

A general buffer of 20-25% beyond the expected delivery interval is recommended for 
the warn threshold. The fail threshold should represent one full missed delivery cycle.

| Cadence | Expected Interval | Warn Threshold | Fail Threshold | Notes |
| --- | --- | --- | --- | --- |
| Daily | 24 hours | 30 hours | 54 hours | Accounts for late morning deliveries |
| Every other day | 48 hours | 58 hours | 96 hours | One full extra cycle before fail |
| Weekly | 168 hours | 192 hours | 216 hours | Warn same day as next expected delivery |
| Bi-weekly | 336 hours | 360 hours | 384 hours | One day buffer each side |
| Monthly | 720 hours | 768 hours | 792 hours | ~2 day buffer beyond expected delivery |
| Ad hoc | N/A | None | None | Set both to None to skip freshness check |

---

> ⚠️ **Important — Align Thresholds With Pipeline Run Time**
>
> Freshness thresholds are measured from the file's last modified timestamp to the 
> moment the pipeline runs. If your pipeline runs at 2:00 PM and the file typically 
> arrives at 8:00 AM, a 24 hour warn threshold is appropriate. If the file sometimes 
> arrives at noon, a 24 hour threshold may fire false positive warnings on days where 
> the file is simply late — not missing. Always factor in both the expected delivery 
> time and the pipeline execution time when setting thresholds. When in doubt, add 
> more buffer rather than less — a warn threshold that fires too often will be ignored.

---

> 💡 **Known Limitation — Modified Timestamp vs Content Change**
>
> The freshness check uses the file's last modified timestamp on disk. If a file is 
> manually saved or touched without a content change, the timestamp resets and the 
> freshness check will pass even if the underlying data has not been updated. This 
> is acceptable for V1 and is standard practice for file-based pipelines. A more 
> robust approach — hashing file contents and comparing against the prior run — is 
> a candidate for a future platform enhancement.

---

- **Source Metadata** — Defines how the source file is delivered and how frequently 
it is expected to refresh. `DELIVERY_METHOD` and `REFRESH_CADENCE` are used to 
populate the source registry on first successful run and inform freshness threshold 
decisions. Valid values are documented inline in the config block.

- **Row Count Thresholds** — Minimum and maximum expected row counts. The pipeline fails 
if the source returns fewer rows than `MIN_ROW_COUNT`. Set `MAX_ROW_COUNT` if the source 
has a known upper bound that would indicate a data quality issue if exceeded.

- **Quantity Columns** — Columns containing numeric quantities that should be rounded up 
to the nearest whole integer before staging. Applies `math.ceil()`. Use this for any 
field where decimal precision is not meaningful or supported by the downstream process.

- **Text Columns** — Columns that must remain as string type regardless of their values. 
Prevents numeric coercion from stripping leading zeros on any identifier field — 
applicable to codes, IDs, account numbers or any field where the string representation 
is the authoritative value.

- **Schema Setup** — Automatically creates all required Lakehouse schemas if they 
do not already exist. Runs on every execution — safe to re-run since `IF NOT EXISTS` 
prevents duplication. No action required from the analyst. Schemas created: `raw`, 
`stg`, `core`, `out`, `cfg`, `log`, `sbx`.

---

> ⚠️ **Important — Source Column Names vs Renamed Column Names**
>
> Several configuration values in this block must reference sanitized column names 
> or renamed column names depending on when they are applied in the pipeline. 
> Using the wrong name will cause silent failures or incorrect behavior that is 
> difficult to trace.
>
> Run `get_sanitized_column_names()` from Section 3 before filling in config for 
> a new source — it prints the exact sanitized names ready to paste directly into 
> `EXPECTED_COLUMNS`, `REQUIRED_COLUMNS` and `COLUMN_RENAMES`.
>
> | Config Field | Use | Reason |
> | --- | --- | --- |
> | `EXPECTED_COLUMNS` | Sanitized source names | Validated against df_raw after sanitization in Section 4 |
> | `REQUIRED_COLUMNS` | Sanitized source names | Validated against df_raw after sanitization in Section 4 |
> | `COLUMN_RENAMES` | Sanitized → renamed | Maps sanitized source names to final standard names |
> | `BUSINESS_KEY` | Renamed names | Applied against df_stg after renames are applied |
> | `TEXT_COLUMNS` | Renamed names | Applied during transformation after renames |
> | `QTY_COLUMNS` | Renamed names | Applied during transformation after renames |

---


In [ ]:
# =============================================================================
# CONFIGURATION — update all values here before running
# =============================================================================

# -- Run metadata
RUN_ID          = str(uuid.uuid4())
LOAD_DATE       = date.today()
LOAD_TIMESTAMP  = datetime.now(pytz.timezone("America/New_York"))

# -- Environment
# -- Set to "dev" during development, "prod" for production execution
# -- Dev writes to sbx_ tables — safe to drop and rebuild freely
# -- Prod writes to governed raw_ and stg_ tables
ENV             = "dev"

# -- Source identity
SOURCE_NAME     = "[source_name]"           # e.g. "johnson_brothers"
DOMAIN          = "[domain]"                # e.g. "cust_inventory"
NOTEBOOK_TYPE   = "source"                  # source | core | output
LAYER_RAW       = "raw"
LAYER_STG       = "stg"

# -- Source file
# CHANGE TO
SRC_PATH        = r"[full_file_path]"        # L drive:    r"L:\VMI Operations\...\file.csv"
                                             # Lakehouse:  "/lakehouse/default/Files/raw/{DOMAIN}/{file}.{SRC_FORMAT}"
SRC_FORMAT      = "[csv | excel | parquet]" # e.g. "csv"
SRC_SHEET       = None                      # Excel only, None if CSV
SRC_ENCODING    = "utf-8"                   # update if source file has encoding issues
SRC_DELIMITER   = ","                       # csv/txt only — update per source
                                            # csv: "," | txt: "|" or other | tab: "\t"
                                            # excel: not used, can leave as ","

# -- Target tables
# -- Auto-generated from DOMAIN and SOURCE_NAME — do not rename manually
# -- ENV controls whether writes go to sandbox or governed production tables
if ENV == "dev":
    RAW_TABLE   = f"sbx.{DOMAIN}_{SOURCE_NAME}_raw"
    STG_TABLE   = f"sbx.{DOMAIN}_{SOURCE_NAME}_stg"
elif ENV == "prod":
    RAW_TABLE   = f"{LAYER_RAW}.{DOMAIN}_{SOURCE_NAME}"
    STG_TABLE   = f"{LAYER_STG}.{DOMAIN}_{SOURCE_NAME}"
else:
    raise ValueError(
        f"Invalid ENV value: '{ENV}'. "
        f"Expected 'dev' or 'prod'. "
        f"Check the ENV variable in the Configuration section."
    )

# -- Schema expectations
# -- Use SANITIZED source column names — run get_sanitized_column_names() in Section 3
# -- to get the exact names to paste here. Validated against df_raw before renames.
EXPECTED_COLUMNS = [
    # "[col1]", "[col2]", "[col3]"
]

# -- Use SANITIZED source column names — subset of EXPECTED_COLUMNS
# -- Must never be null. Validated against df_raw before renames.
REQUIRED_COLUMNS = [
    # "[col1]", "[col2]"
]

# -- Business key
# -- Use RENAMED column names — these are the RIGHT SIDE values from COLUMN_RENAMES
# -- Applied against df_stg after renames. Most common config error is using sanitized
# -- source names here instead of the renamed names.
BUSINESS_KEY    = [
    # "[col1]", "[col2]"
]

# -- Left side:  sanitized source column name (from get_sanitized_column_names() in Section 5)
# -- Right side: domain standard column name — see Section 2 markdown for standard targets.
#                Consistent right-side names across all sources in the same domain
#                are required for the core notebook to union them cleanly.
COLUMN_RENAMES  = {     # format --> "source_name":"rename_target" (ex --> "maxorderqty":"max_order_qty")
    ""              : "location",
    ""              : "item",
    ""              : "max_order_qty",
    ""              : "on_hand_qty",
    ""              : "in_transit_qty"
}

# -- Freshness
# -- Refer to the freshness threshold guide in the Section 2 Markdown for cadence recommendations
# -- Use None for both values to skip freshness check entirely for ad hoc sources
FRESHNESS_WARN_HOURS  = 30       # warn if file is older than this many hours
FRESHNESS_FAIL_HOURS  = 54       # fail pipeline if file is older than this many hours
SOURCE_OWNER          = "[name or team]"    # contact if freshness validation fails

# -- Source metadata
# -- Used to populate the source registry on first successful run
# -- delivery_method: "l_drive" | "lakehouse" | "sftp" | "edi"
# -- refresh_cadence: "daily" | "weekly" | "monthly" | "ad_hoc"
DELIVERY_METHOD  = "[l_drive | lakehouse | sftp | edi]"
REFRESH_CADENCE  = "[daily | weekly | monthly | ad_hoc]"

# -- Row count thresholds
# -- Set MIN_ROW_COUNT based on the smallest valid file you would expect from this source
# -- Set MAX_ROW_COUNT if the source has a known upper bound — use None if no upper bound
MIN_ROW_COUNT   = 1
MAX_ROW_COUNT   = None

# -- Quantity fields
# -- Use RENAMED column names — applied during transformation after renames
QTY_COLUMNS     = [
    # "[qty_col1]"
]

# -- String fields that must stay as text
# -- Use RENAMED column names — applied during transformation after renames
TEXT_COLUMNS    = [
    # "[identifier_col1]", "[identifier_col2]"
]

# -- Schema setup
# -- Creates required Lakehouse schemas if they do not already exist
# -- Safe to re-run — IF NOT EXISTS prevents duplication
required_schemas = ["raw", "stg", "core", "out", "cfg", "log", "sbx"]

for schema in required_schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

print(f"Schemas verified: {required_schemas}")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

## 3 · Helper Functions

This section imports the shared VMI Analytics utility library from the Lakehouse 
Files zone. All platform helper functions are maintained in a single file — 
`Files/utils/vmi_utils.py` — so updates apply automatically to every notebook 
without requiring changes in multiple places.

**To update a helper function:** edit `vmi_utils.py` in the Lakehouse Files zone. 
All notebooks pick up the change on their next run automatically.

**Functions available after import:**

- **`get_current_user()`** — Resolves executing user identity via Fabric context.
- **`get_run_info()`** — Returns current run metadata summary.
- **`log_message()`** — Prints timestamped status messages to notebook output.
- **`check_freshness()`** — Validates source file age against thresholds.
- **`check_columns()`** — Validates DataFrame columns against config expectations.
- **`check_row_count()`** — Validates DataFrame row count against thresholds.
- **`check_nulls()`** — Validates required columns contain no nulls.
- **`add_audit_columns()`** — Appends standard audit columns before Delta write.
- **`log_run()`** — Writes structured run record to log.pipeline_runs.
- **`register_source()`** — Upserts source record in cfg.source_registry.
- **`get_sanitized_column_names()`** — Prints sanitized column names for config setup.

---

> ⚠️ **Important — Shared Library Updates Affect All Notebooks**
>
> Because all notebooks import from the same `vmi_utils.py` file, any change to 
> that file affects every notebook on the next run. Test changes in a dev notebook 
> before updating the shared file. Never edit `vmi_utils.py` directly in production 
> without validating the change first.

---

> 💡 **Tip — Using help() to Inspect Functions**
>
> Every function in vmi_utils includes a docstring. To inspect what a function 
> does, its arguments and usage examples, run `help(function_name)` in any cell.
>
> Example:
> ```
> help(check_freshness)
> help(log_run)
> ```


In [ ]:
# -- Import shared VMI utility functions
import sys
sys.path.insert(0, "/lakehouse/default/Files/utils/")
import vmi_utils
from vmi_utils import *

log_message("Setup", "VMI utility functions loaded successfully.")

StatementMeta(, , -1, SessionStarting, , SessionStarting, True)

## 4 · Extract Source

Reads the source file from the path defined in the configuration block and loads it 
into a pandas DataFrame. This section does not apply any transformation or business 
logic — the goal is simply to get the raw data into memory exactly as it arrived.

Freshness validation is performed here before the file is read. If the file is missing 
or too old, the pipeline fails immediately rather than loading stale data silently.

**What happens in this section:**
- Freshness check is run against the source file path
- File is read into a raw DataFrame using the format defined in config
- A row count is printed immediately after load as a basic sanity check
- No columns are renamed, dropped or modified here — that happens in Section 7

**What does NOT happen in this section:**
- No transformation or cleaning — the raw DataFrame must reflect the source exactly 
  as it arrived, including any nulls, inconsistencies or formatting issues
- No type casting — all columns are read as strings and cast explicitly in Section 7
- No validation of content — structural validation happens in Section 5

**Supported formats:**
- `csv`   — read with `pd.read_csv()`
- `excel` — read with `pd.read_excel()`, requires `SRC_SHEET` to be set in config

---

> ⚠️ **Important — All Columns Are Read as Strings Intentionally**
>
> Every source file is read with `dtype=str` regardless of the data it contains. 
> This is intentional and must not be changed. Allowing pandas to infer data types 
> at read time causes numeric coercion on identifier fields — stripping leading zeros 
> from codes, IDs and account numbers before any validation or logging has occurred. 
> Reading as string ensures the raw DataFrame is a true representation of the source 
> file. All type casting is applied explicitly and visibly in Section 7 where it is 
> documented and traceable. If you believe a column needs a specific type at read 
> time, the answer is to add it to the type casting logic in Section 7 — not to 
> remove `dtype=str` from the read call.

---

**If your source file fails to load:**
- Confirm the file path in config is correct and accessible from Fabric
- Confirm the file format matches `SRC_FORMAT` in config
- Check whether the file requires a specific encoding — update `SRC_ENCODING` if needed
- For Excel files, confirm the sheet name matches `SRC_SHEET` exactly

**Supported formats:**
- `csv`   — read with `pd.read_csv()`, delimiter set by `SRC_DELIMITER` in config
- `txt`   — read with `pd.read_csv()`, update `SRC_DELIMITER` to match source format
- `tab`   — read with `pd.read_csv()`, tab delimiter applied automatically
- `excel` — read with `pd.read_excel()`, requires `SRC_SHEET` to be set in config

**Source file path formats:**
- **Lakehouse Files** — use `/lakehouse/default/Files/...` format. The relative path 
  shown in the Fabric explorer (`Files/raw/...`) does not resolve correctly for pandas 
  reads — always use the full `/lakehouse/default/` prefix.
- **L drive** — use the full UNC path with a raw string prefix: 
  `r"L:\VMI Operations\..."`. Requires gateway access to be configured.
- **SFTP** — not supported in V1 via direct read. Files must be downloaded to the 
  Lakehouse Files zone first before this notebook can read them.


In [ ]:
# -- Freshness check before reading
check_freshness(SRC_PATH, FRESHNESS_WARN_HOURS, FRESHNESS_FAIL_HOURS, SOURCE_OWNER)

# -- Read source file into raw DataFrame
try:
    if SRC_FORMAT == "csv":
        df_raw = pd.read_csv(
            SRC_PATH,
            encoding=SRC_ENCODING,
            dtype=str,
            index_col=False,
            sep=SRC_DELIMITER
        )

    elif SRC_FORMAT == "txt":
        df_raw = pd.read_csv(
            SRC_PATH,
            encoding=SRC_ENCODING,
            dtype=str,
            index_col=False,
            sep=SRC_DELIMITER
        )

    elif SRC_FORMAT == "tab":
        df_raw = pd.read_csv(
            SRC_PATH,
            encoding=SRC_ENCODING,
            dtype=str,
            index_col=False,
            sep="\t"
        )

    elif SRC_FORMAT == "excel":
        df_raw = pd.read_excel(
            SRC_PATH,
            sheet_name=SRC_SHEET,
            dtype=str
        )

    else:
        raise ValueError(
            f"Unsupported source format: '{SRC_FORMAT}'. "
            f"Expected 'csv', 'txt', 'tab' or 'excel'."
        )

    # -- Sanitize column names for Delta compatibility
    # -- Strips whitespace, lowercases, replaces spaces and special characters
    # -- with underscores. This is a platform standard applied to all sources
    # -- at read time. Full semantic renames happen in Section 7.
    df_raw.columns = (
        pd.Index(df_raw.columns)
        .str.strip()
        .str.lower()
        .str.replace(' ', '_', regex=False)
        .str.replace(r'[^a-zA-Z0-9_]', '_', regex=True)
        .str.replace(r'_+', '_', regex=True)
        .str.strip('_')
    )

    log_message("Extract", f"Source file loaded successfully: {SRC_PATH}")
    log_message("Extract", f"Raw row count on load: {len(df_raw):,}")
    log_message("Extract", f"Raw column count on load: {len(df_raw.columns):,}")

except FileNotFoundError as error:
    log_message("Extract", f"Source file not found: {error}", level="ERROR")
    raise

except Exception as error:
    log_message("Extract", f"Failed to read source file: {error}", level="ERROR")
    raise

## 5 · Validate Raw

Validates the raw DataFrame immediately after load, before anything is written to the 
Lakehouse. The goal of this section is to catch structural problems with the source 
file early — before bad data gets persisted to a Delta table and causes silent failures 
downstream.

This section does not fix anything. It only checks and reports. Transformation and 
cleaning happen in Section 7.

**What is validated here:**
- Column structure — expected columns are present, required columns are not missing
- Row count — source returned at least the minimum expected number of rows
- Null values — required fields do not contain nulls in the raw source

**What does NOT happen in this section:**
- No transformation or cleaning — a validation failure means the source has a problem 
  that must be investigated, not fixed inline
- No business logic — this section only validates structure, not content correctness
- No writing to the Lakehouse — nothing is persisted until Section 6

**Validation behavior:**
- A failed required column check will stop the pipeline immediately
- A failed row count check will stop the pipeline immediately
- A failed null check on a required field will stop the pipeline immediately
- Unexpected columns present in the source will log a warning but will not stop the pipeline
- Missing expected (but not required) columns will log a warning but will not stop the pipeline

---

> ⚠️ **Important — Never Modify Thresholds to Force a Pass**
>
> When a validation check fails, the correct response is always to investigate the 
> source — not to lower the threshold until the check passes. A row count below 
> minimum means the source delivered less data than expected. A missing required 
> column means the source schema changed. A null in a required field means the 
> source has a data quality problem. These are all signals that something upstream 
> has gone wrong. Adjusting EXPECTED_COLUMNS, REQUIRED_COLUMNS or MIN_ROW_COUNT 
> to make a failing pipeline pass without understanding why it failed is how silent 
> data quality issues enter the platform and propagate downstream undetected.

---

> 💡 **Tip — Raw Validation Failures Are Faster to Debug Than Downstream Failures**
>
> A pipeline that fails loudly in Section 5 with a clear validation message is 
> significantly easier to diagnose than one that passes silently and produces 
> incorrect output that is only discovered hours or days later in a downstream 
> process. A validation failure here is the pipeline working as intended — treat 
> it as useful information, not as an obstacle.

---

**If validation fails:**
- Check the source file manually to confirm it arrived correctly and is not empty 
  or corrupted
- Confirm EXPECTED_COLUMNS and REQUIRED_COLUMNS in config match the actual source 
  file structure
- Do not modify validation thresholds to make a bad file pass — fix the source 
  or the config


In [ ]:
# -- Validate column structure
check_columns(df_raw, EXPECTED_COLUMNS, REQUIRED_COLUMNS, step="Validate Raw")

# -- Validate row count
check_row_count(df_raw, MIN_ROW_COUNT, MAX_ROW_COUNT, step="Validate Raw")

# -- Validate nulls on required fields
check_nulls(df_raw, REQUIRED_COLUMNS, step="Validate Raw")

log_message("Validate Raw", "All raw validation checks passed.")

## 6 · Write Raw Table

Writes the validated raw DataFrame to a Delta table in the raw schema of the 
VMI Analytics Lakehouse. This is the first governed persistence point in the pipeline.

The raw table is an unmodified snapshot of the source data — the only additions are 
the four audit columns appended here. No business logic, renaming or cleaning is 
applied before this write. The raw table should always reflect exactly what arrived 
from the source.

**What happens in this section:**
- Audit columns are appended to the raw DataFrame
- DataFrame is converted from pandas to a Spark DataFrame for Delta write
- Data is written to the raw Delta table in append mode, partitioned by load_date
- Row count is confirmed after write as a final sanity check

**What does NOT happen in this section:**
- No transformation, cleaning or renaming — the raw table must reflect the source 
  exactly as it arrived. Any modifications belong in Section 7
- No deduplication — the raw table preserves every row from the source including 
  duplicates. Deduplication is a staged layer concern
- No business logic — audit columns are the only addition to the raw DataFrame 
  before this write

**Write behavior:**
- Mode is always append — raw tables are never overwritten
- Data is partitioned by load_date so each day's snapshot is physically isolated
- If the table does not exist yet, Fabric will create it automatically on first run
- If the table already exists, today's partition is appended alongside prior dates

---

> ⚠️ **Important — Never Use Overwrite Mode to Resolve a Write Failure**
>
> If the raw table write fails, the correct response is to investigate the root 
> cause — not to change the write mode to overwrite. Switching to overwrite deletes 
> all prior partitions and destroys the historical audit trail the raw table exists 
> to provide. Common causes of write failures are schema mismatches between today's 
> source and a prior run, permission issues or a missing raw schema in the Lakehouse. 
> All of these have correct solutions that do not involve overwriting data. If a 
> specific partition must be removed due to a confirmed data error, that requires 
> a deliberate Delta table maintenance operation — consult the platform owner before 
> taking that action.

---

> 💡 **Debugging Tip — Compare Pandas and Spark Row Counts**
>
> The row count logged after the write reflects the pandas DataFrame row count — 
> the same count that was validated in Section 5. If you ever suspect a write 
> produced a different number of rows than expected, query the Delta table directly 
> and filter by the current pipeline_run_id to confirm the actual rows written match 
> the pandas count. A divergence between the two is rare but is a signal worth 
> investigating before the pipeline is considered complete.

---

**If the write fails:**
- Confirm the raw schema exists in the Lakehouse
- Confirm the notebook has write access to the Lakehouse
- Check whether a schema mismatch exists between today's source and a prior run — 
  this usually means a column was added or removed at the source
- Do not change write mode to overwrite to resolve errors — investigate the root cause


In [ ]:
# -- Append audit columns before writing
df_raw = add_audit_columns(df_raw, SRC_PATH, RUN_ID, LOAD_DATE, LOAD_TIMESTAMP)

log_message("Write Raw", f"Audit columns appended. Writing to {RAW_TABLE}...")

# -- Convert pandas DataFrame to Spark DataFrame for Delta write
spark_df_raw = spark.createDataFrame(df_raw)

# -- Write to raw Delta table
try:
    (spark_df_raw.write
        .format("delta")
        .mode("append")
        .partitionBy("load_date")
        .saveAsTable(RAW_TABLE)
    )

    log_message("Write Raw", 
        f"Raw table write complete. "
        f"Rows written: {df_raw.shape[0]:,} | "
        f"Table: {RAW_TABLE} | "
        f"Partition: {LOAD_DATE}"
    )

except Exception as error:
    log_message("Write Raw", f"Raw table write failed: {error}", level="ERROR")
    raise

## 7 · Transform & Standardize

Applies all cleaning, standardization and business logic to produce the staged 
DataFrame. This section and Section 2 (Configuration) are the only sections that 
should require meaningful changes when repurposing this template for a new source.

Transformations are applied in a deliberate order — renaming first, then type casting, 
then null handling, then deduplication. Do not reorder these steps. Renaming before 
casting ensures all subsequent steps reference standard column names. Casting before 
null handling ensures nulls are evaluated against the correct data types.

**What happens in this section:**
- Source columns are renamed to platform snake_case standards using COLUMN_RENAMES
- Text columns are explicitly cast to string to prevent leading zero stripping
- Quantity columns are ceiling-rounded to the nearest whole integer before staging
- All string columns are stripped of leading and trailing whitespace
- Null values in non-required fields are filled with appropriate defaults
- Records are deduplicated against the defined BUSINESS_KEY
- Rejected duplicate records are counted and logged with a reason code

**What does NOT happen in this section:**
- No joins to other tables or sources — a staged table represents one source only.
  Cross-source joins and crosswalk resolution happen in the core layer notebook.
- No process-specific output formatting — column selection, ordering and final naming 
  for any downstream delivery file happen in the output layer notebook.
- No inclusion or exclusion decisions — filters, business rules and exclusion logic 
  are core and output layer concerns, not staging concerns.
- No aggregation or summarization — the staged table preserves the same grain as 
  the raw source. Aggregation happens in the core layer if needed.

**Layer responsibility reminder:**

| Layer | Responsibility |
| --- | --- |
| `raw_` | Exact source snapshot — no changes except audit columns |
| `stg_` | THIS SECTION — clean and standardize one source |
| `core_` | Join sources, resolve crosswalks, apply business rules |
| `out_` | Format and deliver the final output file |

---

> ⚠️ **Important — Do Not Reorder the Transformation Steps**
>
> The seven transformation steps in this section must be executed in the order 
> they are written. The order is not arbitrary — each step depends on the state 
> produced by the step before it. Renaming must happen first so all subsequent 
> steps reference standard column names rather than raw source names. Type casting 
> must happen before null handling so nulls are evaluated against the correct data 
> types. Null handling must happen before deduplication so deduplication operates 
> on clean values. Reordering these steps will produce incorrect or unpredictable 
> results that may not surface as an obvious error — they will produce silently 
> wrong data.

---

> ⚠️ **Important — All Source-Specific Logic Belongs in Step 6**
>
> The template provides a clearly marked block in Step 6 for source-specific 
> transformations. Any logic that only applies to one source — field derivations, 
> conditional mappings, custom filtering, format corrections — belongs there and 
> nowhere else. Do not insert source-specific steps between the standard 
> transformation steps above it. Keeping source-specific logic isolated means 
> the standard transformation structure remains consistent and readable across 
> every source notebook built from this template.

---

**Transformation principles:**
- Every transformation must be traceable to a business rule or platform standard
- No silent row drops — rejected records are always counted and logged
- No hardcoded values — all source-specific rules belong in Section 2 (Configuration)
- If a transformation does not have a documented reason, it should not be here

**Adding source-specific logic:**
- Additional transformations for a specific source go in the clearly marked block 
  in Step 6 of the code cell below
- Keep source-specific logic isolated from standard transformations so the template 
  structure remains readable and consistent across all source notebooks


In [ ]:
# -- Start with a copy of the raw DataFrame — never mutate df_raw directly
df_stg = df_raw.copy()

# =============================================================================
# STEP 1 — COLUMN RENAMES
# Rename source columns to platform snake_case standards defined in config.
# All downstream steps reference the renamed columns.
# =============================================================================
if COLUMN_RENAMES:
    df_stg = df_stg.rename(columns=COLUMN_RENAMES)
    log_message("Transform", f"Columns renamed: {COLUMN_RENAMES}")
else:
    log_message("Transform", "No column renames defined in config — skipping.")

# =============================================================================
# STEP 2 — FORCE TEXT COLUMNS TO STRING
# Prevents numeric coercion from stripping leading zeros on customer IDs,
# SKU codes and account identifiers. Applied before any other type casting.
# =============================================================================
for column in TEXT_COLUMNS:
    if column in df_stg.columns:
        df_stg[column] = df_stg[column].astype(str).str.strip()
        log_message("Transform", f"Column '{column}' cast to string and stripped.")
    else:
        log_message("Transform",
            f"TEXT_COLUMNS references '{column}' but column not found — skipping.",
            level="WARN")

# =============================================================================
# STEP 3 — STRIP WHITESPACE FROM ALL STRING COLUMNS
# Removes leading and trailing spaces from confirmed string columns only.
# Skips columns that have already been cast to numeric types.
# Trailing spaces in identifier fields are a known source of join failures —
# this step ensures they cannot propagate into the staged table.
# =============================================================================
string_columns = [
    col for col in df_stg.columns
    if df_stg[col].dtype == object
]

for col in string_columns:
    try:
        df_stg[col] = df_stg[col].str.strip()
    except AttributeError:
        log_message("Transform",
            f"Column '{col}' skipped in whitespace strip — not a string type.",
            level="WARN")

log_message("Transform",
    f"Whitespace stripped from {len(string_columns)} string columns.")

# =============================================================================
# STEP 4 — CEILING ROUND QUANTITY COLUMNS
# Rounds quantity fields up to the nearest whole integer before staging.
# Use this for any field where decimal precision is not meaningful or
# supported by the downstream process receiving this data.
# =============================================================================
for column in QTY_COLUMNS:
    if column in df_stg.columns:
        df_stg[column] = (
            pd.to_numeric(df_stg[column], errors="coerce")
              .apply(lambda value: math.ceil(value) if pd.notna(value) else value)
        )
        log_message("Transform",
            f"Column '{column}' ceiling-rounded to nearest whole integer.")
    else:
        log_message("Transform",
            f"QTY_COLUMNS references '{column}' but column not found — skipping.",
            level="WARN")

# =============================================================================
# STEP 5 — NULL HANDLING
# Fills nulls in non-required fields with appropriate defaults.
# Required fields are validated in Section 5 — if they are null here,
# the pipeline should have already failed. Zero-fill for quantity fields
# ensures downstream processes receive explicit zero records rather than
# blank values that may be interpreted differently per system.
# =============================================================================
for column in QTY_COLUMNS:
    if column in df_stg.columns:
        df_stg[column] = df_stg[column].fillna(0)

log_message("Transform", "Null handling applied to quantity fields.")

# =============================================================================
# STEP 6 — SOURCE-SPECIFIC TRANSFORMATIONS
# Add any logic specific to this source below.
# Keep source-specific steps clearly commented with a business rule rationale.
# Do not add generic platform logic here — that belongs in the steps above.
# =============================================================================

# -- /// [SOURCE-SPECIFIC LOGIC GOES HERE] /// --


# =============================================================================
# STEP 7 — DEDUPLICATION
# Removes duplicate records based on the BUSINESS_KEY defined in config.
# Rejected records are counted and logged — no silent row drops.
# =============================================================================
if BUSINESS_KEY:
    row_count_before = len(df_stg)

    df_stg = df_stg.sort_values(
        by=BUSINESS_KEY + ["load_timestamp"],
        ascending=True
    ).drop_duplicates(
        subset=BUSINESS_KEY,
        keep="last"
    )

    row_count_after  = len(df_stg)
    duplicates_dropped = row_count_before - row_count_after

    if duplicates_dropped > 0:
        log_message("Transform",
            f"Deduplication removed {duplicates_dropped:,} records. "
            f"Business key: {BUSINESS_KEY} | "
            f"Rows before: {row_count_before:,} | "
            f"Rows after: {row_count_after:,}",
            level="WARN")
    else:
        log_message("Transform",
            f"Deduplication check passed — no duplicates found on business key: {BUSINESS_KEY}")
else:
    log_message("Transform",
        "No BUSINESS_KEY defined in config — deduplication skipped.", level="WARN")

log_message("Transform",
    f"Transformation complete. Staged row count: {len(df_stg):,}")

StatementMeta(, , -1, SessionStarting, , SessionStarting, True)

## 8 · Validate Staged

Validates the staged DataFrame after all transformations have been applied, before 
anything is written to the Lakehouse. This is the final quality gate before governed 
data is persisted to the stg_ Delta table.

Where Section 5 validated the raw source structure, this section validates the 
business quality of the transformed result. A record that passed raw validation 
can still fail staged validation if transformations produced unexpected nulls, 
introduced duplicates or resulted in an unexpected row count.

**What happens in this section:**
- Required fields are checked for nulls after transformation
- Business key uniqueness is confirmed — no duplicate keys should remain after 
  deduplication in Section 7
- Row count is validated against expected thresholds
- A staged row count summary is printed for manual review

**What does NOT happen in this section:**
- No transformations or fixes — if staged validation fails, the root cause belongs 
  in Section 7, not here
- No comparison against other sources or prior runs — cross-source validation 
  happens in the core layer
- No output formatting checks — column structure for downstream delivery is 
  validated in the output layer notebook

**Validation behavior:**
- A failed required field null check will stop the pipeline immediately
- A duplicate business key after deduplication indicates a logic error in Section 7 
  and will stop the pipeline immediately
- A row count below the minimum threshold will stop the pipeline immediately
- Warnings are logged for unexpected conditions that do not warrant a full stop

---

> ⚠️ **Important — Duplicate Business Keys After Deduplication**
>
> If this section fails on a duplicate business key check, the problem is never the 
> data — it is the BUSINESS_KEY definition in Section 2 (Configuration). A duplicate 
> key surviving deduplication means the key does not uniquely identify a row in this 
> source and needs additional columns added to it. Do not attempt to fix this by 
> modifying the deduplication logic in Section 7. Go back to Section 2, redefine 
> the BUSINESS_KEY, and re-run from Section 7 forward.

---

> 💡 **Tip — Staged Validation Failures Point to Section 7, Not the Source**
>
> A failure in this section means the transformation logic in Section 7 produced 
> an unexpected result — not that the source file is bad. If the source file had 
> a structural problem, Section 5 would have caught it already. When staged 
> validation fails, start your investigation in Section 7 by reviewing the 
> transformation steps in order. Check whether a rename broke a downstream 
> reference, whether a type cast introduced unexpected nulls or whether 
> deduplication operated on the wrong columns. Do not go back to the source 
> file until Section 7 has been ruled out.

---

**If staged validation fails:**
- Do not modify thresholds to force a pass — investigate the transformation logic 
  in Section 7
- A null in a required field after transformation usually means the source delivered 
  unexpected data or a column rename in config is incorrect
- Duplicate business keys after deduplication indicate an incomplete BUSINESS_KEY 
  definition in Section 2 — see callout above
- Row count failures after transformation usually indicate an unintended row drop 
  in Section 7 — review deduplication and null handling logic


In [ ]:
# -- Validate required fields are not null after transformation
check_nulls(df_stg, REQUIRED_COLUMNS, step="Validate Staged")

# -- Validate row count is within expected thresholds
check_row_count(df_stg, MIN_ROW_COUNT, MAX_ROW_COUNT, step="Validate Staged")

# -- Validate business key uniqueness after deduplication
if BUSINESS_KEY:
    duplicate_count = df_stg.duplicated(subset=BUSINESS_KEY).sum()

    if duplicate_count > 0:
        raise ValueError(
            f"Staged validation failed — {duplicate_count:,} duplicate business keys "
            f"remain after deduplication. "
            f"Business key: {BUSINESS_KEY}. "
            f"Review deduplication logic in Section 7."
        )
    else:
        log_message("Validate Staged",
            f"Business key uniqueness check passed — no duplicates on: {BUSINESS_KEY}")
else:
    log_message("Validate Staged",
        "No BUSINESS_KEY defined in config — uniqueness check skipped.",
        level="WARN")

# -- Print staged summary for manual review
log_message("Validate Staged",
    f"Staged row count: {len(df_stg):,} | "
    f"Column count: {len(df_stg.columns):,}"
)

log_message("Validate Staged", "All staged validation checks passed.")

## 9 · Write Staged Table

Writes the validated staged DataFrame to a Delta table in the stg_ schema of the 
VMI Analytics Lakehouse. This is the second and final governed persistence point 
in this notebook.

The staged table represents a cleaned, standardized and deduplicated version of 
a single source. It is the authoritative input for all downstream core layer 
notebooks that join, enrich or aggregate this source alongside others.

**What happens in this section:**
- Audit columns are refreshed to reflect the staged write timestamp
- DataFrame is converted from pandas to a Spark DataFrame for Delta write
- Data is written to the stg Delta table in append mode, partitioned by load_date
- Row count is confirmed after write

**What does NOT happen in this section:**
- No additional transformations — if something still needs fixing it belongs in 
  Section 7
- No joins or enrichment — the staged table contains one source only
- No output formatting — the staged table is not the final delivery artifact, 
  it is an intermediate governed asset for reuse across multiple downstream processes

**Write behavior:**
- Mode is always append — staged tables are never overwritten
- Data is partitioned by load_date so each day's snapshot is physically isolated
- If the table does not exist yet, Fabric will create it automatically on first run
- If the table already exists, today's partition is appended alongside prior dates

---

> ⚠️ **Important — Never Overwrite a Staged Table to Fix Bad Data**
>
> If a transformation error is discovered after a staged table has already been 
> written, do not rerun the notebook in overwrite mode to correct it. Overwriting 
> breaks the append history and removes the audit trail. The correct approach is 
> to fix the transformation logic in Section 7, rerun the notebook, and allow the 
> corrected records to land as a new load_date partition. If a bad partition must 
> be removed, that requires a deliberate Delta table maintenance operation — not 
> a silent overwrite. Consult the platform owner before deleting any partition 
> from a governed table.

---

**If the write fails:**
- Confirm the stg schema exists in the Lakehouse
- Confirm the notebook has write access to the Lakehouse
- Check whether a schema mismatch exists between today's staged output and a 
  prior run — this usually means a column was added or removed in Section 7
- Do not change write mode to overwrite to resolve errors — investigate the 
  root cause


In [ ]:
# -- Append audit columns before writing
df_stg = add_audit_columns(df_stg, SRC_PATH, RUN_ID, LOAD_DATE, LOAD_TIMESTAMP)

log_message("Write Staged", f"Audit columns appended. Writing to {STG_TABLE}...")

# -- Convert pandas DataFrame to Spark DataFrame for Delta write
spark_df_stg = spark.createDataFrame(df_stg)

# -- Write to staged Delta table
try:
    (spark_df_stg.write
        .format("delta")
        .mode("append")
        .partitionBy("load_date")
        .saveAsTable(STG_TABLE)
    )

    log_message("Write Staged",
        f"Staged table write complete. "
        f"Rows written: {df_stg.shape[0]:,} | "
        f"Table: {STG_TABLE} | "
        f"Partition: {LOAD_DATE}"
    )

except Exception as error:
    log_message("Write Staged", f"Staged table write failed: {error}", level="ERROR")
    raise

## 10 · Log Run Status

Writes a structured run record to the log.pipeline_runs Delta table upon successful 
completion of the notebook. This section only executes if all prior sections completed 
without error — a failure anywhere upstream will bypass this section and fall through 
to Section 11 (Exception Handling), which handles failure logging.

Every notebook execution — success or failure — must produce a log entry. This is 
non-negotiable per platform standards. The log table is the operational record of 
the platform and is the first place to look when investigating a pipeline issue.

**What happens in this section:**
- A structured run record is written to log.pipeline_runs
- Final row counts for both the raw and staged tables are recorded
- The executing user identity is captured and recorded
- Run status is recorded as SUCCESS
- A final confirmation message is printed to the notebook output

**What does NOT happen in this section:**
- Failure logging does not happen here — that is handled in Section 11
- No data validation or transformation — this section assumes all prior sections 
  passed successfully
- No notification or alerting — those capabilities are a future platform feature

**What is recorded in the log entry:**

| Field | Description |
| --- | --- |
| `pipeline_run_id` | Unique ID tying this log entry to every record written in this run |
| `notebook_type` | Type of notebook — source, core or output |
| `load_date` | The date this pipeline executed |
| `load_timestamp` | The timestamp this pipeline executed in US/Eastern time |
| `source_name` | The source identifier from config |
| `raw_table` | The raw Delta table written — null for core and output notebooks |
| `stg_table` | The staged Delta table written — null for core and output notebooks |
| `core_table` | The core Delta table written — null for source notebooks |
| `out_table` | The output Delta table written — null for source notebooks |
| `status` | SUCCESS or FAILURE |
| `rows_raw` | Rows written to raw — 0 if not applicable |
| `rows_stg` | Rows written to stg — 0 if not applicable |
| `rows_core` | Rows written to core — 0 if not applicable |
| `rows_out` | Rows written to output — 0 if not applicable |
| `executed_by` | The user or service account that triggered this pipeline run |
| `message` | Empty on success — populated with error detail on failure |

---

> ⚠️ **Important — The Log Table Is Your Operational Record**
>
> The log.pipeline_runs table is the authoritative record of every pipeline execution 
> on the platform. Do not skip logging, comment it out during testing or allow the 
> notebook to complete silently without a log entry. If you are testing and do not 
> want to pollute the log table, use a clearly named sbx_ notebook instead. A 
> production notebook that does not log is unauditable and untraceable when something 
> goes wrong.

---

> 💡 **Tip — The Log Table Is the First Place to Look When Something Goes Wrong**
>
> Before re-running a failed pipeline or digging into notebook output, query 
> log.pipeline_runs first. Filter by load_date or pipeline_run_id to find the 
> relevant run. The status, rows_raw, rows_stg and message fields will tell you 
> whether the failure occurred before or after the writes, who ran the pipeline 
> and what the error was. This is faster and more reliable than scrolling through 
> notebook output — especially for scheduled runs where the output may not be 
> immediately accessible.
>
> Example query:
> ```sql
> SELECT *
> FROM log.pipeline_runs
> WHERE load_date = current_date()
> ORDER BY load_timestamp DESC
> ```

---

**If the log write fails:**
- A log write failure should still raise an error — do not suppress it silently
- Confirm the log schema exists in the Lakehouse
- Confirm log.pipeline_runs exists — on first run this table will be created 
  automatically by the log_run() function
- A log write failure does not mean the raw and staged writes failed — check 
  those tables independently before rerunning


In [ ]:
# -- Write success log entry
try:
    log_run(
        spark          = spark,
        mssparkutils   = mssparkutils, 
        source_name    = SOURCE_NAME,
        notebook_type  = NOTEBOOK_TYPE,
        run_id         = RUN_ID,
        load_date      = LOAD_DATE,
        load_timestamp = LOAD_TIMESTAMP,
        raw_table      = RAW_TABLE,
        stg_table      = STG_TABLE,
        status         = "SUCCESS",
        rows_raw       = df_raw.shape[0],
        rows_stg       = df_stg.shape[0]
    )

    register_source(
        spark            = spark,
        mssparkutils     = mssparkutils,
        source_name      = SOURCE_NAME,
        domain           = DOMAIN,
        notebook_type    = NOTEBOOK_TYPE,
        src_path         = SRC_PATH,
        src_format       = SRC_FORMAT,
        delivery_method  = DELIVERY_METHOD,
        refresh_cadence  = REFRESH_CADENCE,
        expected_columns = EXPECTED_COLUMNS,
        business_key     = BUSINESS_KEY,
        min_row_count    = MIN_ROW_COUNT,
        max_row_count    = MAX_ROW_COUNT,
        source_owner     = SOURCE_OWNER,
        load_timestamp   = LOAD_TIMESTAMP,
        run_id           = RUN_ID,
        rows_raw         = df_raw.shape[0],
        rows_stg         = df_stg.shape[0]
    )

    executed_by = get_current_user(mssparkutils)

    log_message("Log Run",
        f"Pipeline completed successfully. "
        f"Source: {SOURCE_NAME} | "
        f"Notebook type: {NOTEBOOK_TYPE} | "
        f"Raw rows: {df_raw.shape[0]:,} | "
        f"Staged rows: {df_stg.shape[0]:,} | "
        f"Run ID: {RUN_ID} | "
        f"Executed by: {executed_by}"
    )

except Exception as error:
    log_message("Log Run",
        f"Log write failed — pipeline completed but run was not recorded: {error}",
        level="ERROR")
    raise

StatementMeta(, , -1, SessionStarting, , SessionStarting, True)

## 11 · Exception Handling

Catches any unhandled error that occurs during pipeline execution, logs a structured 
failure record and re-raises the error so the notebook and any parent pipeline show 
a visible failure. This section wraps the entire notebook execution and is the 
safety net for anything that was not caught by earlier validation steps.

A failed pipeline must always be visible. Suppressing errors, catching exceptions 
silently or allowing a notebook to complete with a green status despite a failure 
is not acceptable behavior on this platform. Failures must be loud, logged and 
traceable.

**What happens in this section:**
- Any unhandled exception is caught
- A failure log entry is written to log.pipeline_runs with the error detail
- The full stack trace is printed to the notebook output for debugging
- The error is re-raised so Fabric marks the notebook as failed
- Row counts recorded in the failure log reflect what was actually written —
  if the failure occurred before the raw write, both counts will be zero

**What does NOT happen in this section:**
- No data recovery or retry logic — retries are handled at the pipeline 
  orchestration level, not inside the notebook
- No partial success logging — a notebook either succeeds fully or fails fully
- No suppression of errors — the error is always re-raised after logging

**Failure log behavior:**
- Status is recorded as FAILURE in log.pipeline_runs
- The error message and full stack trace are captured in the message field
- Row counts reflect actual writes — zero if failure occurred before writes

---

> ⚠️ **Important — Always Re-Raise After Logging**
>
> It may be tempting to catch an exception, log it and allow the notebook to 
> complete without raising the error — particularly during development when a 
> green notebook status feels more convenient. Do not do this. A notebook that 
> completes successfully despite a failure will be treated as a success by any 
> parent pipeline, meaning downstream notebooks will run against incomplete or 
> missing data with no warning. The failure log entry alone is not sufficient — 
> Fabric must also see the notebook as failed. Always re-raise.

---

> 💡 **Debugging Tip — Reading the Failure Log**
>
> When a pipeline fails, the first step is always to query log.pipeline_runs 
> filtered by the pipeline_run_id or load_date of the failed run. The message 
> field contains the exception type and the full stack trace captured at the 
> time of failure. This is faster than re-running the notebook and gives you 
> the exact line and error type without needing to reproduce the failure.
>
> Example query:
> ```sql
> SELECT *
> FROM log.pipeline_runs
> WHERE load_date = current_date()
> AND status = 'FAILURE'
> ORDER BY load_timestamp DESC
> ```

---

**If the failure log write itself fails:**
- The original error will still be re-raised
- Check log.pipeline_runs schema for compatibility issues
- A failure log write failure does not change the fact that the notebook failed —
  investigate the original error first, then address the log write separately


In [ ]:
# -- Wrap entire notebook in exception handler
# -- This cell should be the last cell in the notebook
# -- Variables may not exist if failure occurred early — use dir() check
# -- for safety when referencing counts in the log entry

try:
    pass

except Exception as error:

    full_stack_trace = traceback.format_exc()


    log_message("Exception Handler",
        f"Unhandled exception caught. "
        f"Source: {SOURCE_NAME if 'SOURCE_NAME' in dir() else 'unknown'} | "
        f"Notebook type: {NOTEBOOK_TYPE if 'NOTEBOOK_TYPE' in dir() else 'unknown'} | "
        f"Executed by: {executed_by} | "
        f"Error: {error}",
        level="ERROR")
    print(full_stack_trace)

    try:
        log_run(
            spark          = spark,
            source_name    = SOURCE_NAME if "SOURCE_NAME" in dir() else "unknown",
            notebook_type  = NOTEBOOK_TYPE if "NOTEBOOK_TYPE" in dir() else "unknown",
            run_id         = RUN_ID if "RUN_ID" in dir() else "unknown",
            load_date      = LOAD_DATE if "LOAD_DATE" in dir() else date.today(),
            load_timestamp = LOAD_TIMESTAMP if "LOAD_TIMESTAMP" in dir() else datetime.now(),
            raw_table      = RAW_TABLE if "RAW_TABLE" in dir() else None,
            stg_table      = STG_TABLE if "STG_TABLE" in dir() else None,
            status         = "FAILURE",
            rows_raw       = df_raw.shape[0] if "df_raw" in dir() else 0,
            rows_stg       = df_stg.shape[0] if "df_stg" in dir() else 0,
            message        = f"{str(error)} | {full_stack_trace}"
        )
    except Exception as log_error:
        log_message("Exception Handler",
            f"Failure log write also failed: {log_error}",
            level="ERROR")

    raise error

log_message("Complete",
    f"Notebook execution complete. "
    f"ENV: {ENV} | "
    f"Notebook type: {NOTEBOOK_TYPE} | "
    f"Source: {SOURCE_NAME} | "
    f"Run ID: {RUN_ID}"
)